# Mapping launch sites and their surroundings

## Interactive proximity analysis with Folium

Landing success may depend not only on the rocket and payload but also on where the launch site sits and what is nearby. This notebook builds an interactive Folium map of the launch sites and measures the distance from a site to the coastline, roads and the nearest city.

### Objectives

- Mark every launch site on a map
- Colour-code individual launches by success or failure
- Measure distances from a site to nearby coastline, railway, highway and city

Import the mapping and data libraries.

In [ ]:
!pip3 install folium
!pip3 install pandas


In [62]:
import folium
import pandas as pd


In [63]:
# Import folium MarkerCluster plugin
from folium.plugins import MarkerCluster
# Import folium MousePosition plugin
from folium.plugins import MousePosition
# Import folium DivIcon plugin
from folium.features import DivIcon

## Marking the launch sites

Add each site's coordinates to the map.

The `spacex_launch_geo.csv` table augments each launch record with latitude and longitude.

In [64]:
# Source path is configurable so the notebook can run offline.
GEO_CSV = "../data/processed/spacex_launch_geo.csv"
spacex_df = pd.read_csv(GEO_CSV)


Inspect the site coordinates.

In [65]:
# Select relevant sub-columns: `Launch Site`, `Lat(Latitude)`, `Long(Longitude)`, `class`
spacex_df = spacex_df[['Launch Site', 'Lat', 'Long', 'class']]
launch_sites_df = spacex_df.groupby(['Launch Site'], as_index=False).first()
launch_sites_df = launch_sites_df[['Launch Site', 'Lat', 'Long']]
launch_sites_df

,Launch Site,Lat,Long
0,CCAFS LC-40,28.562302,-80.577356
1,CCAFS SLC-40,28.563197,-80.576820
2,KSC LC-39A,28.573255,-80.646895
3,VAFB SLC-4E,34.632834,-120.610745


Plain coordinates are hard to interpret; plotting them on a map is far more informative.

Create a Folium map centred on the NASA Johnson Space Center.

In [66]:
# Start location is NASA Johnson Space Center
nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map = folium.Map(location=nasa_coordinate, zoom_start=10)

Use `folium.Circle` to draw a highlighted area with a text label at a coordinate.

In [67]:
# Create a blue circle at NASA Johnson Space Center's coordinate with a popup label showing its name
circle = folium.Circle(nasa_coordinate, radius=1000, color='#d35400', fill=True).add_child(folium.Popup('NASA Johnson Space Center'))
# Create a blue circle at NASA Johnson Space Center's coordinate with a icon showing its name
marker = folium.map.Marker(
    nasa_coordinate,
    # Create an icon as a text label
    icon=DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % 'NASA JSC',
        )
    )
site_map.add_child(circle)
site_map.add_child(marker)

A small circle appears near Houston; zoom in to see it.

Add a circle and label for every launch site.

Create a `folium.Circle` and `folium.Marker` for each site.

Example `folium.Circle`:

`folium.Circle(coordinate, radius=1000, color='#000000', fill=True).add_child(folium.Popup(...))`

Example `folium.Marker`:

`folium.map.Marker(coordinate, icon=DivIcon(icon_size=(20,20), icon_anchor=(0,0), html=...))`

In [68]:


site_map = folium.Map(location=nasa_coordinate, zoom_start=5)

for index, site in launch_sites_df.iterrows():
    coordinates = [site['Lat'], site['Long']]
    
    # Circle with popup
    circle = folium.Circle(
        coordinates, radius=1000, 
        color='#d35400', fill=True,
        popup=site['Launch Site']
    )
    
    # Marker with text label
    marker = folium.Marker(
        coordinates,
        icon=DivIcon(
            icon_size=(20,20),
            icon_anchor=(0,0),
            html='<div style="font-size: 12px; color:#d35400;"><b>%s</b></div>' % site['Launch Site'],
        )
    )
    
    site_map.add_child(circle)
    site_map.add_child(marker)
site_map

The resulting map marks all four launch sites.

Zoom in and explore: are the sites near the Equator? Are they on the coast? What infrastructure is nearby?

## Marking successful and failed launches

Enhance the map by colour-coding each launch by outcome, so high-success sites stand out.

In [69]:
spacex_df.tail(10)

,Launch Site,Lat,Long,class
46,KSC LC-39A,28.573255,-80.646895,1
47,KSC LC-39A,28.573255,-80.646895,1
48,KSC LC-39A,28.573255,-80.646895,1
49,CCAFS SLC-40,28.563197,-80.576820,1
50,CCAFS SLC-40,28.563197,-80.576820,1
51,CCAFS SLC-40,28.563197,-80.576820,0
52,CCAFS SLC-40,28.563197,-80.576820,0
53,CCAFS SLC-40,28.563197,-80.576820,0
54,CCAFS SLC-40,28.563197,-80.576820,1
55,CCAFS SLC-40,28.563197,-80.576820,0


Create one marker per launch: green for a successful landing (`class=1`), red for a failed one (`class=0`).

Many launches share the same site coordinate, so marker clustering keeps the map readable.

Create a `MarkerCluster`.

In [70]:
marker_cluster = MarkerCluster()


Add a `marker_color` column derived from `class`.

In [71]:
# Colours are derived from the launch outcome; see the helper below.


In [72]:
# Function to assign color to launch outcome
def assign_marker_color(launch_outcome):
    if launch_outcome == 1:
        return 'green'
    else:
        return 'red'
    
spacex_df['marker_color'] = spacex_df['class'].apply(assign_marker_color)
spacex_df.tail(10)

,Launch Site,Lat,Long,class,marker_color
46,KSC LC-39A,28.573255,-80.646895,1,green
47,KSC LC-39A,28.573255,-80.646895,1,green
48,KSC LC-39A,28.573255,-80.646895,1,green
49,CCAFS SLC-40,28.563197,-80.576820,1,green
50,CCAFS SLC-40,28.563197,-80.576820,1,green
51,CCAFS SLC-40,28.563197,-80.576820,0,red
52,CCAFS SLC-40,28.563197,-80.576820,0,red
53,CCAFS SLC-40,28.563197,-80.576820,0,red
54,CCAFS SLC-40,28.563197,-80.576820,1,green
55,CCAFS SLC-40,28.563197,-80.576820,0,red


Add a `folium.Marker` for each launch to the cluster.

In [73]:
# Add marker_cluster to the current site_map
site_map.add_child(marker_cluster)

# Add one coloured marker per launch record to the cluster
for index, record in spacex_df.iterrows():
    coordinates = [record['Lat'], record['Long']]
    marker = folium.Marker(
        location=coordinates,
        popup=record['Launch Site'],
        icon=folium.Icon(color=record['marker_color'])
    )
    marker_cluster.add_child(marker)

site_map


The updated map shows clustered, colour-coded launches per site.

From the colour-coded clusters you can see which sites have the higher success rates.

## Distances to nearby features

Next, measure how close each site is to the coastline, roads and the nearest city.

Add a `MousePosition` control to read coordinates directly from the map.

In [74]:
# Add Mouse Position to get the coordinate (Lat, Long) for a mouse over on the map
formatter = "function(num) {return L.Util.formatNum(num, 5);};"
mouse_position = MousePosition(
    position='topright',
    separator=' Long: ',
    empty_string='NaN',
    lng_first=False,
    num_digits=20,
    prefix='Lat:',
    lat_formatter=formatter,
    lng_formatter=formatter,
)

site_map.add_child(mouse_position)
site_map

Zoom into a site and use the mouse-position readout to note the coordinates of nearby features such as coastline, railway or highway.

The distance between two points is computed from their latitude and longitude using the great-circle formula below.

In [75]:
from math import sin, cos, sqrt, atan2, radians

def calculate_distance(lat1, lon1, lat2, lon2):
    # approximate radius of earth in km
    R = 6373.0

    lat1 = radians(lat1)
    lon1 = radians(lon1)
    lat2 = radians(lat2)
    lon2 = radians(lon2)

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    distance = R * c
    return distance

Measure the distance from the site to the closest point on the coastline.

In [76]:
# find coordinate of the closet coastline
# e.g.,: Lat: 28.56367  Lon: -80.57163
# 34.632834	-120.610745
coastline_lat = 34.632834
coastline_lon =  -120.6267
launch_site_lat = 34.632834
launch_site_lon = -120.610745
distance_coastline = calculate_distance(launch_site_lat, launch_site_lon, coastline_lat, coastline_lon)



Add a marker showing the computed distance.

In [77]:
# Create and add a folium.Marker on your selected closest coastline point on the map
# Display the distance between coastline point and launch site using the icon property 
# for example
coordinate = [coastline_lat, coastline_lon]
distance_marker = folium.Marker(
[34.6328,-120.62461],
icon=DivIcon(
    icon_size=(20,20),
    icon_anchor=(0,0),
    html='<div style="font-size: 12; color:#d35400;"><b>Coastline %s</b></div>' % "{:10.2f} KM".format(distance_coastline),
    )
)
site_map.add_child(distance_marker)

Draw a `PolyLine` between the site and the coastline point.

In [78]:
# Create a `folium.PolyLine` object using the coastline coordinates and launch site coordinate
coordinates = [coordinate,[launch_site_lat, launch_site_lon]]
lines=folium.PolyLine(locations=coordinates, weight=1)
site_map.add_child(lines)

The map now shows the distance line to the coast.

Repeat the same approach for the nearest city, railway and highway.

Railway map symbol:

Highway map symbol:

City map symbol:

In [79]:
# Create a marker with distance to a closest city
# Draw a line between the marker to the launch site

city_lat = 34.638
city_lon = -120.47639
distance_city = calculate_distance(launch_site_lat, launch_site_lon, city_lat, city_lon)
coordinate = [city_lat, city_lon]
distance_marker = folium.Marker(
    [34.6333, -120.5982],
    icon=DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),
        html='<div style="font-size: 12; color:#d35400;"><b> City %s</b></div>' % "{:10.2f} KM".format(distance_city),
        )
    )
site_map.add_child(distance_marker) 
lines=folium.PolyLine(locations=[coordinate,[launch_site_lat, launch_site_lon]], weight=1)
site_map.add_child(lines)
    

In [80]:
# Create a marker with distance to a closest railway
# Draw a line between the marker to the launch site

railway_lat = 34.638
railway_lon = -120.6231
distance_railway = calculate_distance(launch_site_lat, launch_site_lon, railway_lat, railway_lon)
coordinate = [railway_lat, railway_lon]
distance_marker = folium.Marker(
    [34.6366, -120.62],
    icon=DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),
        html='<div style="font-size: 12; color:#d35400;"><b> Railway %s</b></div>' % "{:10.2f} KM".format(distance_railway),
        )
    )
site_map.add_child(distance_marker)
lines=folium.PolyLine(locations=[coordinate,[launch_site_lat, launch_site_lon]], weight=1)
site_map.add_child(lines)

In [81]:
# Create a marker with distance to a closest highway 
# Draw a line between the marker to the launch site

highway_lat = 34.63944
highway_lon = -120.620
distance_highway = calculate_distance(launch_site_lat, launch_site_lon, highway_lat, highway_lon)
coordinate = [highway_lat, highway_lon]
distance_marker = folium.Marker(
    [34.6392, -120.6199],
    icon=DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),
        html='<div style="font-size: 12; color:#d35400;"><b> Highway %s</b></div>' % "{:10.2f} KM".format(distance_highway),
        )
    )
site_map.add_child(distance_marker)
lines=folium.PolyLine(locations=[coordinate,[launch_site_lat, launch_site_lon]], weight=1)
site_map.add_child(lines)

With the distance lines drawn, the proximity conclusions can be read off directly:

- Sites are close to railways: yes
- Close to highways: yes
- Close to the coastline: yes
- Close to a city: mostly yes

### Next steps

The interactive map shows how location and proximity shape launch operations. The same site data feeds the dashboard and the interactive map committed under `reports/maps/`.

## Summary

- Mapped all launch sites and individual outcomes
- Measured distances to coastline, railway, highway and city
- Produced an interactive map for the report